# Static demand, side by side: your code $\rightarrow$ `PyBLP`

*Practical session, BLP (1995) — companion to the problem set.*

In the problem set you built the aggregate random-coefficient logit estimator **by hand**:
market-share integration, the BLP contraction mapping, and a nested GMM loop for $\sigma$.

Here we do two things:

1. **Validate** — feed your *own* simulated data into [`PyBLP`](https://pyblp.readthedocs.io)
   (Conlon & Gortmaker 2020) and check it reproduces your hand-rolled estimates.
2. **Go beyond** what is painful to code by hand:
   **optimal instruments**, **elasticities & diversion ratios**, and a **merger simulation**.

The model is identical to the problem set:
$$u_{ijt}=\underbrace{x_{jt}\beta+\alpha p_{jt}+\xi_{jt}}_{\delta_{jt}}+\underbrace{x_{jt}^{1}\,\sigma\,\nu_i}_{\mu_{ijt}}+\varepsilon_{ijt},
\qquad \nu_i\sim N(0,1),\ \varepsilon\sim\text{T1EV}.$$

In [1]:
import functools
import numpy as np
import pandas as pd
import scipy
import pyblp

# Your problem-set modules — unchanged:
from simulation import simulate_data
from integration import quadrature_hermite
from model import supply_foc, compute_shares, compute_mean_utility
from estimation import two_sls_formula

pyblp.options.verbose = False
np.set_printoptions(precision=4, suppress=True)
print("pyblp", pyblp.__version__)

pyblp 1.2.0


## Part A — Regenerate *your* data

Same data-generating process and true parameters as the problem set
($\beta=(2,2)$, $\alpha=-2$, $\sigma=(0,1)$ — i.e. a random coefficient on $x^1$ only).

The Bertrand–Nash prices are solved **market by market**: with single-product firms each
market's pricing FOC only involves that market's own products, so the equilibrium is found
one market at a time.

In [2]:
params = {
    "gamma_1": [0.7, 0.7], "gamma_2": [1, 1, 1],
    "beta": [2, 2], "alpha": -2,
    "sigma_c": np.array([[1, 0.7], [0.7, 1]]), "sigma": [0, 1],
}
num_markets, num_products = 250, 10
quad_draws, quad_weights = quadrature_hermite(n_quad_points=20, mu=0, sigma=1)


# Bertrand-Nash prices solved one market at a time (single-product firms).
def bertrand_by_market(df, params, qd, qw, J, T):
    mc = df["marginal_costs"].to_numpy()
    x0, x1, xi = (df[c].to_numpy() for c in ["obs_char_0", "obs_char_1", "xi"])
    prices = np.empty(T * J)
    for t in range(T):
        s = slice(t * J, (t + 1) * J)
        foc = functools.partial(supply_foc, params=params, x_0=x0[s], x_1=x1[s], xi=xi[s],
                                marginal_costs=mc[s], num_markets=1, num_products=J,
                                quad_draws=qd, quad_weights=qw)
        prices[s] = scipy.optimize.fsolve(foc, mc[s])
    return prices


df = simulate_data(params=params, num_products=num_products, num_markets=num_markets, seed=100)
df["prices"] = bertrand_by_market(df, params, quad_draws, quad_weights, num_products, num_markets)

mean_utility = compute_mean_utility(params, df["obs_char_0"].to_numpy(), df["obs_char_1"].to_numpy(),
                                    df["prices"].to_numpy(), df["xi"].to_numpy())
df["shares"] = compute_shares(params, df["obs_char_0"].to_numpy(), df["obs_char_1"].to_numpy(),
                              mean_utility, num_markets, num_products,
                              quad_draws, quad_weights).reshape(num_products * num_markets)
df.head()

obs_char_0  obs_char_1  obs_cost_shifter_1  \
market product                                               
0      0                 1    1.543405            0.249526   
       1                 1    1.278369            0.267269   
       2                 1    1.424518            0.621049   
       3                 1    1.844776            0.150104   
       4                 1    1.004719            0.391014   

                obs_cost_shifter_2  obs_cost_shifter_3        xi     omega  \
market product                                                               
0      0                  0.439082            0.940456 -1.092337 -0.577983   
       1                  0.713405            0.164054  0.561220 -0.261381   
       2                  0.785709            0.495000  0.988689  1.020035   
       3                  0.780814            0.246959  2.028907  2.356624   
       4                  0.516103            0.681441  0.483138 -0.448913   

                marginal_costs    prices    shares  
market product                                      
0      0              2.831465  3.353475  0.030079  
       1              2.478206  3.077605  0.139972  
       2              4.618956  5.122303  0.005161  
       3              5.525845  6.032745  0.007325  
       4              2.542949  3.082218  0.068080

## Part B — Reshape into a `PyBLP` `product_data` frame

`PyBLP` expects long-format data with reserved column names:

- `market_ids`, `firm_ids` (single-product firms $\Rightarrow$ the product index *is* the firm id),
- `shares`, `prices`,
- excluded instruments `demand_instruments0, demand_instruments1, ...`
  (the exogenous regressors in $X_1$ are added to the instrument set automatically).

We use the same instruments as in your problem set: the **BLP instrument** (sum of rivals'
$x^1$), the polynomial term $(x^1)^2$, and the three **cost shifters** $w_1,w_2,w_3$.

In [3]:
d = (df.reset_index()
       .rename(columns={"market": "market_ids", "product": "firm_ids", "obs_char_1": "x1"}))
d["blp_instr"] = d.groupby("market_ids")["x1"].transform("sum") - d["x1"]
d["x1_sq"] = d["x1"] ** 2
for k, col in enumerate(["x1_sq", "blp_instr",
                         "obs_cost_shifter_1", "obs_cost_shifter_2", "obs_cost_shifter_3"]):
    d[f"demand_instruments{k}"] = d[col]
# for k, col in enumerate(["x1_sq", "blp_instr"]):
#     d[f"demand_instruments{k}"] = d[col]

assert (d.groupby("market_ids")["shares"].sum() < 1).all(), "inside shares must leave room for outside good"
d[["market_ids", "firm_ids", "shares", "prices", "x1"]].head()

,market_ids,firm_ids,shares,prices,x1
0,0,0,0.030079,3.353475,1.543405
1,0,1,0.139972,3.077605,1.278369
2,0,2,0.005161,5.122303,1.424518
3,0,3,0.007325,6.032745,1.844776
4,0,4,0.068080,3.082218,1.004719


## Part C — Validate (1): plain logit via 2SLS

With $\sigma=0$ the model is the plain logit and $\ln s_{jt}-\ln s_{0t}=x_{jt}\beta+\alpha p_{jt}+\xi_{jt}$
is linear. We run **our** `two_sls_formula` and **`PyBLP`'s** logit on the same data.

In [4]:
# --- your problem-set 2SLS ---
log_s0 = np.log(1 - d.groupby("market_ids")["shares"].transform("sum"))
y = np.log(d["shares"].to_numpy()) - log_s0
X = d[["obs_char_0", "x1", "prices"]].to_numpy()
Z = d[["obs_char_0", "x1", "x1_sq", "blp_instr",
       "obs_cost_shifter_1", "obs_cost_shifter_2", "obs_cost_shifter_3"]].to_numpy()
# Z = d[["obs_char_0", "x1", "x1_sq", "blp_instr"]].to_numpy()
beta_handrolled, se_handrolled = two_sls_formula(y, X, Z)

# --- PyBLP logit (sigma = 0) ---
logit_problem = pyblp.Problem(pyblp.Formulation("1 + x1 + prices"), d)
logit_results = logit_problem.solve()

comp = pd.DataFrame({
    "your 2SLS":  beta_handrolled,
    "pyblp logit": logit_results.beta.flatten(),
    "truth":      [2, 2, -2],
}, index=["const", "x1", "price"])
comp

,your 2SLS,pyblp logit,truth
const,1.025467,1.020723,2
x1,2.339266,2.339775,2
price,-1.932190,-1.931757,-2


## Part D — Validate (2): random coefficient $\sigma$

Now the full model. `PyBLP` does what we coded: invert shares for $\delta$ via the
contraction, then nest that in a GMM search over $\sigma$ — but with better inner-loop solvers and analytic gradients.

The nonlinear part is one random coefficient on `x1`, so `X2 = Formulation('0 + x1')`.

In [5]:
problem = pyblp.Problem(
    product_formulations=(pyblp.Formulation("1 + x1 + prices"),  # X1: mean utility (delta)
                          pyblp.Formulation("0 + x1")),          # X2: random coefficient
    product_data=d,
    integration=pyblp.Integration("product", 9),   # Gauss-Hermite, like your quadrature
)
results = problem.solve(sigma=0.5, optimization=pyblp.Optimization("l-bfgs-b"))
print("beta :", results.beta.flatten(), "   (truth: 2, 2, -2)")
print("sigma:", results.sigma.flatten(), "  se:", results.sigma_se.flatten(), "   (truth: 1)")
results

beta : [ 1.7491  2.0935 -1.9837]    (truth: 2, 2, -2)
sigma: [0.9188]   se: [0.2315]    (truth: 1)


Problem Results Summary:
GMM     Objective      Projected       Reduced     Clipped  Weighting Matrix  Covariance Matrix
Step      Value      Gradient Norm     Hessian     Shares   Condition Number  Condition Number 
----  -------------  -------------  -------------  -------  ----------------  -----------------
 2    +3.613949E+00  +8.737217E-10  +3.815895E+01     0      +4.966772E+05      +7.525780E+03  

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective   Fixed Point  Contraction
   Time      Converged   Iterations   Evaluations  Iterations   Evaluations
-----------  ---------  ------------  -----------  -----------  -----------
 00:00:05       Yes          8            14          18027        57750   

Nonlinear Coefficient Estimates (Robust SEs in Parentheses):
Sigma:        x1       
------  ---------------
  x1     +9.187692E-01 
        (+2.314988E-01)

Beta Estimates (Robust SEs in Parentheses):
       1               x1             prices     
----------

## Part E — Optimal instruments

Approximating the Chamberlain (1987) optimal instruments by hand is fiddly; in `PyBLP` it is one call.
We recompute instruments at the first-stage estimate and re-solve.

In [6]:
oi = results.compute_optimal_instruments(method="approximate")
opt_results = oi.to_problem().solve(sigma=results.sigma, optimization=pyblp.Optimization("l-bfgs-b"))

pd.DataFrame({
    "sigma":    [results.sigma.flatten()[0],    opt_results.sigma.flatten()[0]],
    "sigma se": [results.sigma_se.flatten()[0], opt_results.sigma_se.flatten()[0]],
}, index=["BLP instruments", "optimal instruments"])

,sigma,sigma se
BLP instruments,0.918769,0.231499
optimal instruments,1.041261,0.120461


The point estimate moves toward the truth ($\sigma=1$) and the standard error shrinks.

## Part F — Elasticities & diversion ratios

With `PyBLP`, one call each; we look at market 0.

In [7]:
elasticities = results.compute_elasticities()      # d log s_j / d log p_k
diversion    = results.compute_diversion_ratios()   # D_{j->k}

own = results.extract_diagonals(elasticities)
print(f"mean own-price elasticity: {np.nanmean(own):.2f}")

market0 = d["market_ids"].to_numpy() == 0
print("\nElasticity matrix, market 0 (rows j, cols k):")
print(np.round(elasticities[market0][:, :num_products], 3))

mean own-price elasticity: -7.06

Elasticity matrix, market 0 (rows j, cols k):
[[ -6.38    1.055   0.068   0.132   0.461   0.708   0.004   0.439   0.543
    0.367]
 [  0.247  -5.109   0.063   0.114   0.453   0.684   0.004   0.382   0.524
    0.332]
 [  0.261   1.029 -10.095   0.124   0.458   0.698   0.004   0.413   0.535
    0.351]
 [  0.301   1.116   0.074 -11.813   0.466   0.73    0.005   0.511   0.561
    0.409]
 [  0.222   0.931   0.058   0.098  -5.673   0.654   0.003   0.329   0.5
    0.296]
 [  0.232   0.959   0.06    0.105   0.447  -3.581   0.004   0.351   0.51
    0.311]
 [  0.284   1.081   0.071   0.141   0.464   0.718 -13.65    0.469   0.552
    0.384]
 [  0.299   1.112   0.074   0.153   0.466   0.728   0.005  -6.259   0.56
    0.406]
 [  0.234   0.962   0.06    0.106   0.447   0.669   0.004   0.354  -4.152
    0.313]
 [  0.275   1.061   0.069   0.134   0.462   0.711   0.004   0.447   0.545
   -7.401]]


## Part G — Merger simulation

Recover marginal costs from the estimated demand + Bertrand FOCs, then re-solve equilibrium
prices after firms 0 and 1 merge (they internalise diversion between their products).

In [8]:
costs = results.compute_costs()
merged_firm_ids = d["firm_ids"].replace({1: 0}).to_numpy()  # firm 1 absorbed into firm 0
prices_post = results.compute_prices(firm_ids=merged_firm_ids, costs=costs)

chg = prices_post.flatten() - d["prices"].to_numpy()
print(f"mean price change, all products : {chg.mean():+.4f}")
print(f"mean price change, merging firms: {chg[np.isin(d['firm_ids'], [0, 1])].mean():+.4f}")
print(f"mean price change, rival firms  : {chg[~np.isin(d['firm_ids'], [0, 1])].mean():+.4f}")

mean price change, all products : +0.0104
mean price change, merging firms: +0.0505
mean price change, rival firms  : +0.0003
